[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # W_q, W_k, W_v, W_o
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        # Q from x_q, K/V from x_kv, no causal mask
        B, S_q, D = x_q.shape
        _, S_kv, _ = x_kv.shape

        # project to Q, K, V
        Q = self.W_q(x_q) # (B, S_q, D)
        K = self.W_k(x_kv) # (B, S_kv, D)
        V = self.W_v(x_kv) # (B, S_kv, D)

        # reshape for multi-head: (B, S, D) -> (B, num_heads, S, head_dim)
        Q = Q.view(B, S_q, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S_kv, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S_kv, self.num_heads, self.head_dim).transpose(1, 2)

        # scaled dot-product attention (B, num_heads, S, S)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        # (B, num_heads, S, head_dim)
        attn = nn.functional.softmax(scores, dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, S_q, D)

        return self.W_o(out)
        

In [4]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.0ms)
  ✅ [2/4] Q and KV different lengths (3.0ms)
  ✅ [3/4] No causal mask — all KV affects all Q (24.9ms)
  ✅ [4/4] Gradient flow (2.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (31.7ms total)
  Progress saved. Run status() to see your dashboard.

